In [1]:
import torch
import random
import sys
import os
import re
import numpy as np


In [2]:
Skf_File = os.path.join("../corrupt_pp", "C-C.skf")
Result_File = os.path.join("..","detailed.out")

N_SAMPLES=10
LOWER = -0.6
UPPER = 0.0

In [3]:
def modify_skf(skf_file, output_file, Ep, Es, SPE):
    with open(skf_file, "r") as file:
            lines = file.readlines()
    line = lines[1]
    parameters = list(re.finditer( r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", line ))
    replacements= [f"{Ep:.10f}", f"{Es:.10f}", f"{SPE:.10f}"]

    for parameter, replacement in reversed(list(zip(parameters[1:4], replacements))): 
          start, end = parameter.span()
          line = line[:start] + replacement + line[end:]
    lines[1] = line
    print("New Random ", Ep, ", ", Es, ", ",SPE)    
    with open(output_file, "w") as f:
          f.writelines(lines)

In [4]:
import dataset
input = []
output = []
for i in range(N_SAMPLES):
    Ep = random.uniform(LOWER, UPPER)
    Es = random.uniform(LOWER, UPPER)
    SPE = random.uniform(LOWER, UPPER)
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE)
    energy, force = dataset.test_md()
    input_data = np.array([Ep, Es, SPE])    
    input.append(input_data)
    output_data = np.array([energy, force])
    output.append(output_data)
input_tensor = torch.tensor(input, dtype=torch.float64)
output_tensor = torch.tensor(output, dtype=torch.float64)


    

New Random  -0.012192271932759624 ,  -0.3528042534888263 ,  -0.5607329057703807
Force RMSE: 1.681809 eV/Å
Energy RMSE: 3049.643505 eV
New Random  -0.3154338872531968 ,  -0.1942729016556491 ,  -0.42602783201088307
Force RMSE: 0.724848 eV/Å
Energy RMSE: 3007.870331 eV
New Random  -0.14580717437740587 ,  -0.5457247875019164 ,  -0.20643500099814832
Force RMSE: 0.289845 eV/Å
Energy RMSE: 3019.776375 eV
New Random  -0.5677152466398844 ,  -0.5467271949590943 ,  -0.23886541667995875
Force RMSE: 2.124726 eV/Å
Energy RMSE: 2917.267007 eV
New Random  -0.09810073918652906 ,  -0.023049568456488867 ,  -0.28441311113486556
Force RMSE: 1.399152 eV/Å
Energy RMSE: 3051.103325 eV
New Random  -0.07679017071270777 ,  -0.05440713145720455 ,  -0.2617492657434466
Force RMSE: 1.484977 eV/Å
Energy RMSE: 3053.224678 eV
New Random  -0.5281328671924966 ,  -0.3714826032618507 ,  -0.3208564778868957
Force RMSE: 1.625690 eV/Å
Energy RMSE: 2940.452366 eV
New Random  -0.13999040162093057 ,  -0.27115627225672073 ,  -0.3

/tmp/ipykernel_113808/285206072.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  input_tensor = torch.tensor(input, dtype=torch.float64)


In [5]:
print("Input Tensor: ", input_tensor)
print("Output Tensor: ", output_tensor)


Input Tensor:  tensor([[-0.0122, -0.3528, -0.5607],
        [-0.3154, -0.1943, -0.4260],
        [-0.1458, -0.5457, -0.2064],
        [-0.5677, -0.5467, -0.2389],
        [-0.0981, -0.0230, -0.2844],
        [-0.0768, -0.0544, -0.2617],
        [-0.5281, -0.3715, -0.3209],
        [-0.1400, -0.2712, -0.3667],
        [-0.4958, -0.4889, -0.0876],
        [-0.5669, -0.1853, -0.2415]], dtype=torch.float64)
Output Tensor:  tensor([[3.0496e+03, 1.6818e+00],
        [3.0079e+03, 7.2485e-01],
        [3.0198e+03, 2.8984e-01],
        [2.9173e+03, 2.1247e+00],
        [3.0511e+03, 1.3992e+00],
        [3.0532e+03, 1.4850e+00],
        [2.9405e+03, 1.6257e+00],
        [3.0370e+03, 1.0222e+00],
        [2.9461e+03, 1.4217e+00],
        [2.9309e+03, 1.5688e+00]], dtype=torch.float64)


In [6]:
# BO
bounds = torch.tensor([[LOWER, UPPER], [LOWER, UPPER], [LOWER, UPPER]], dtype=torch.float64)
def normalize(x):
    return (x - bounds[:, 0]) / (bounds[:, 1] - bounds[:, 0])
def denormalize(x):
    return x * (bounds[:, 1] - bounds[:, 0]) + bounds[:, 0]

train_x = normalize(input_tensor)


In [7]:
scale = torch.tensor([1.0, 1.0], dtype=torch.float64)

def calculate_loss(outputs):
    # Calculate the loss
    loss = torch.sum((outputs/scale), dim=1, keepdim=True)
    return loss

L = calculate_loss(output_tensor)

train_obj = -L

In [8]:
import os
import torch
import warnings
from typing import Optional

from botorch import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.acquisition.objective import GenericMCObjective

from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.transforms.input import Normalize

from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition import qLogExpectedImprovement, qLogNoisyExpectedImprovement

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

# build model
def generate_model(train_x, train_obj, state_dict = None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        input_transform=Normalize(d=train_x.shape[-1]),
        ).to(train_x)

    model = ModelListGP(model_obj)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    print("model is generated")
    return mll, model

mll, model = generate_model(train_x, train_obj)


/home/basic-user/src/pyc/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model is generated


In [9]:
def opimization_observation(acq_func, model, train_x, train_obj, num_restarts=10, raw_samples=512):
    # Optimize acquisition function
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=BadInitialCandidatesWarning)
        new_x, acq_value = optimize_acqf(
            acq_function=acq_func,
            bounds=torch.stack([torch.zeros(train_x.shape[-1], dtype=dtype), torch.ones(train_x.shape[-1], dtype=dtype)]),
            q=1,
            num_restarts=num_restarts,
            raw_samples=raw_samples,
        )
    new_x = new_x.detach()
    new_x_denormalized = denormalize(new_x)
    print("New x: ", new_x_denormalized)
    Ep, Es, SPE = new_x_denormalized[0].tolist()
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE)
    energy, force = dataset.test_md()
    input_data = np.array([Ep, Es, SPE])    
    output_data = np.array([energy, force])
    input_tensor_new = torch.tensor(input_data.reshape(1,-1), dtype=torch.float64)
    output_tensor_new = torch.tensor(output_data.reshape(1,-1), dtype=torch.float64)
    train_x_new = torch.cat((train_x, normalize(input_tensor_new)), dim=0)
    train_obj_new = torch.cat((train_obj, -calculate_loss(output_tensor_new)), dim=0)
    print("New train_x: ", input_tensor_new)
    print("New train_obj: ", calculate_loss(output_tensor_new))
    return train_x_new, train_obj_new

new_x, new_obj = opimization_observation(qLogExpectedImprovement(model=model, best_f=train_obj.max()), model, train_x, train_obj)
mll_new, model_new = generate_model(new_x, new_obj)

New x:  tensor([[-0.6000, -0.4014, -0.1935]], dtype=torch.float64)
New Random  -0.6 ,  -0.4014238320187222 ,  -0.19347306661693536
Force RMSE: 2.091575 eV/Å
Energy RMSE: 2911.047904 eV
New train_x:  tensor([[-0.6000, -0.4014, -0.1935]], dtype=torch.float64)
New train_obj:  tensor([[2913.1395]], dtype=torch.float64)
model is generated


In [10]:
print(new_x, new_obj)

tensor([[0.9797, 0.4120, 0.0654],
        [0.4743, 0.6762, 0.2900],
        [0.7570, 0.0905, 0.6559],
        [0.0538, 0.0888, 0.6019],
        [0.8365, 0.9616, 0.5260],
        [0.8720, 0.9093, 0.5638],
        [0.1198, 0.3809, 0.4652],
        [0.7667, 0.5481, 0.3889],
        [0.1737, 0.1852, 0.8540],
        [0.0552, 0.6911, 0.5974],
        [0.0000, 0.3310, 0.6775]], dtype=torch.float64) tensor([[-3051.3253],
        [-3008.5952],
        [-3020.0662],
        [-2919.3917],
        [-3052.5025],
        [-3054.7097],
        [-2942.0781],
        [-3037.9986],
        [-2947.5256],
        [-2932.4261],
        [-2913.1395]], dtype=torch.float64)


In [11]:
for i in range(10):
    new_x, new_obj = opimization_observation(qLogExpectedImprovement(model=model_new, best_f=train_obj.max()), model, new_x, new_obj)
    mll_new, model_new = generate_model(new_x, new_obj)
    print(denormalize(new_x))

New x:  tensor([[-0.6000, -0.4664, -0.2115]], dtype=torch.float64)
New Random  -0.6 ,  -0.46638847244520865 ,  -0.21154236894752554
Force RMSE: 2.223506 eV/Å
Energy RMSE: 2908.534882 eV
New train_x:  tensor([[-0.6000, -0.4664, -0.2115]], dtype=torch.float64)
New train_obj:  tensor([[2910.7584]], dtype=torch.float64)
model is generated
tensor([[-0.0122, -0.3528, -0.5607],
        [-0.3154, -0.1943, -0.4260],
        [-0.1458, -0.5457, -0.2064],
        [-0.5677, -0.5467, -0.2389],
        [-0.0981, -0.0230, -0.2844],
        [-0.0768, -0.0544, -0.2617],
        [-0.5281, -0.3715, -0.3209],
        [-0.1400, -0.2712, -0.3667],
        [-0.4958, -0.4889, -0.0876],
        [-0.5669, -0.1853, -0.2415],
        [-0.6000, -0.4014, -0.1935],
        [-0.6000, -0.4664, -0.2115]], dtype=torch.float64)
New x:  tensor([[-0.6000, -0.4617, -0.2118]], dtype=torch.float64)
New Random  -0.6 ,  -0.4617322877173445 ,  -0.21178245151480812
Force RMSE: 2.214499 eV/Å
Energy RMSE: 2908.732670 eV
New train_x:

In [12]:
dataset.test_md('../trans3d-0-1/')

Force RMSE: 0.174091 eV/Å
Energy RMSE: 3015.685247 eV


(np.float64(3015.685247181095), np.float64(0.17409102181624272))